# 06 — Benchmarks Comparison

Full benchmark comparison across all three RUL models on CMAPSS FD001–FD004.

Topics:
1. Train all models and record metrics
2. Comparative bar charts (RMSE, MAE, NASA Score)
3. Prediction trajectory visualisation per model
4. Model efficiency: parameters vs. performance
5. Summary table

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

np.random.seed(42)

In [ ]:
from datasets.cmapss_loader import CMAPSSLoader
from models.lstm_predictive import LSTMPredictiveModel
from models.transformer_rul import TransformerRULModel
from models.tcn_model import TCNModel
from evaluation.rul_metrics import rmse, mae, nasa_score


def train_and_evaluate(ModelClass, config, X_tr, y_tr, X_val, y_val, X_test, y_test):
    """Train one model and return metrics dict."""
    model = ModelClass(config)
    n_params = sum(p.numel() for p in model.network.parameters() if p.requires_grad)
    
    t0 = time.time()
    model.fit(X_tr, y_tr, X_val, y_val)
    train_time = time.time() - t0
    
    preds = model.predict(X_test)
    return {
        'rmse':       rmse(y_test, preds),
        'mae':        mae(y_test, preds),
        'nasa_score': nasa_score(y_test, preds),
        'n_params':   n_params,
        'train_s':    train_time,
        'predictions': preds,
    }

In [ ]:
# Load CMAPSS FD001
loader = CMAPSSLoader(
    subset='FD001',
    data_dir='../datasets/data',
    max_rul=125,
    window_size=30,
    normalize=True,
)

try:
    X_train, y_train, X_test, y_test = loader.load()
    data_ok = True
except FileNotFoundError:
    print('CMAPSS data not found — using synthetic data.')
    X_train = np.random.randn(8000, 30, 14).astype(np.float32)
    y_train = np.random.uniform(0, 125, 8000).astype(np.float32)
    X_test  = np.random.randn(100, 30, 14).astype(np.float32)
    y_test  = np.random.uniform(0, 125, 100).astype(np.float32)
    data_ok = False

n_val = int(len(X_train) * 0.1)
idx = np.random.permutation(len(X_train))
X_tr, y_tr = X_train[idx[n_val:]], y_train[idx[n_val:]]
X_val, y_val = X_train[idx[:n_val]], y_train[idx[:n_val]]

N_FEATURES = X_tr.shape[-1]
print(f'Ready: train={X_tr.shape}, val={X_val.shape}, test={X_test.shape}')

## 1. Train All Models

In [ ]:
BASE_CFG = {
    'input_size': N_FEATURES,
    'batch_size': 256,
    'epochs': 30,    # quick demo — use 100 for full benchmark
    'patience': 10,
    'weight_decay': 1e-5,
}

model_specs = {
    'LSTM': (
        LSTMPredictiveModel,
        {**BASE_CFG, 'hidden_size': 128, 'num_layers': 2, 'dropout': 0.2, 'lr': 1e-3}
    ),
    'Transformer': (
        TransformerRULModel,
        {**BASE_CFG, 'd_model': 128, 'n_heads': 8, 'n_layers': 3,
         'dim_feedforward': 256, 'dropout': 0.1, 'lr': 5e-4}
    ),
    'TCN': (
        TCNModel,
        {**BASE_CFG, 'num_channels': [64, 64, 128, 128], 'kernel_size': 3, 'dropout': 0.2, 'lr': 1e-3}
    ),
}

results = {}
for name, (ModelClass, cfg) in model_specs.items():
    print(f'\nTraining {name}...')
    results[name] = train_and_evaluate(
        ModelClass, cfg, X_tr, y_tr, X_val, y_val, X_test, y_test
    )
    print(f'  RMSE={results[name]["rmse"]:.2f}  '
          f'MAE={results[name]["mae"]:.2f}  '
          f'NASA={results[name]["nasa_score"]:.1f}  '
          f'({results[name]["train_s"]:.0f}s)')

## 2. Results Summary Table

In [ ]:
summary = pd.DataFrame({
    name: {
        'RMSE': f"{r['rmse']:.2f}",
        'MAE': f"{r['mae']:.2f}",
        'NASA Score': f"{r['nasa_score']:.1f}",
        'Params': f"{r['n_params']/1000:.0f}K",
        'Train Time': f"{r['train_s']:.0f}s",
    }
    for name, r in results.items()
}).T

print('\nBenchmark Results — CMAPSS FD001 (30 epochs)\n')
print(summary.to_string())
summary

## 3. Comparative Bar Charts

In [ ]:
model_names = list(results.keys())
colors = ['steelblue', 'coral', 'forestgreen']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_to_plot = [
    ('rmse',       'RMSE ↓',        'RMSE (cycles)'),
    ('mae',        'MAE ↓',         'MAE (cycles)'),
    ('nasa_score', 'NASA Score ↓',  'NASA Score'),
]

for ax, (key, title, ylabel) in zip(axes, metrics_to_plot):
    vals = [results[m][key] for m in model_names]
    bars = ax.bar(model_names, vals, color=colors, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{v:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    best_idx = np.argmin(vals)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(2.5)

fig.suptitle('Model Comparison — CMAPSS FD001  (30 epochs)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Prediction Trajectories

In [ ]:
# Sort test units by true RUL for clearer visualisation
sort_idx = np.argsort(y_test)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test[sort_idx], 'k--', linewidth=2, label='True RUL', zorder=10)

for name, color in zip(model_names, colors):
    preds = results[name]['predictions'][sort_idx]
    ax.plot(preds, color=color, linewidth=1.2, alpha=0.8, label=f'{name} (RMSE={results[name]["rmse"]:.1f})')

ax.fill_between(
    range(len(y_test)),
    y_test[sort_idx] - 20, y_test[sort_idx] + 20,
    alpha=0.05, color='gray', label='±20 cycle band'
)

ax.set_xlabel('Test unit (sorted by true RUL)')
ax.set_ylabel('RUL (cycles)')
ax.set_title('RUL Prediction Trajectories — All Models')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 5. Efficiency: Parameters vs. RMSE

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for name, color in zip(model_names, colors):
    r = results[name]
    ax.scatter(
        r['n_params'] / 1000, r['rmse'],
        s=200, color=color, zorder=5, alpha=0.9
    )
    ax.annotate(
        name,
        xy=(r['n_params'] / 1000, r['rmse']),
        xytext=(8, 4), textcoords='offset points',
        fontsize=11, fontweight='bold'
    )

ax.set_xlabel('Model parameters (K)')
ax.set_ylabel('RMSE (cycles)')
ax.set_title('Efficiency: Parameters vs. RMSE')
plt.tight_layout()
plt.show()

## Summary and Recommendations

| Model | Strength | When to use |
|-------|----------|-------------|
| **LSTM** | Robust, interpretable via attention | Default choice; good on most datasets |
| **Transformer** | Best accuracy, parallelisable | GPU available; longer sequences |
| **TCN** | Fewest parameters, fast inference | Edge deployment, latency-constrained |
| **Autoencoder** | No fault labels needed | New equipment with no history |

To reproduce the full benchmark results:
```bash
python benchmarks/run_benchmarks.py --all-datasets --all-models --epochs 100
```